# Event Features Test Notebook

This notebook demonstrates how to extract and display event features for midfielders
from a match, including subtype-based features.


## Setup and Imports


In [1]:
import pandas as pd
from pathlib import Path
import sys

# Add src to path
sys.path.insert(0, str(Path.cwd() / "src"))

from movement_dna.event_features import (
    extract_event_features,
    extract_event_features_from_file,
    extract_obr_events,
    extract_po_events,
    extract_pp_events,
    extract_obe_events,
    extract_events_by_type,
)


## Configuration


In [2]:
# Paths to data files
OPENDATA_DIR = Path.cwd().parent / "opendata" / "data" / "matches"
MATCH_ID = 1886347

# File paths
events_file = OPENDATA_DIR / str(MATCH_ID) / f"{MATCH_ID}_dynamic_events.csv"

# Load physical aggregates to get midfielder IDs and names
physical_aggregates_file = (
    Path.cwd().parent
    / "opendata"
    / "data"
    / "aggregates"
    / "aus1league_physicalaggregates_20242025_midfielders.csv"
)

print(f"Events file: {events_file}")
print(f"Events file exists: {events_file.exists()}")


Events file: /Users/arvid/development/personal/analytics_cup/opendata/data/matches/1886347/1886347_dynamic_events.csv
Events file exists: True


## Load Midfielder Information


In [3]:
def get_midfielder_info():
    """Get midfielder IDs and names from physical aggregates."""
    if physical_aggregates_file.exists():
        df = pd.read_csv(physical_aggregates_file)
        # Filter to midfielders
        midfielders = df[df["position_group"] == "Midfield"]
        # Drop duplicates on player_id to ensure unique index
        # Keep first occurrence if a player appears multiple times
        unique_midfielders = (
            midfielders[["player_id", "player_name", "team_name"]]
            .drop_duplicates(subset=["player_id"], keep="first")
            .set_index("player_id")
        )
        return unique_midfielders.to_dict("index")
    return {}


midfielder_info = get_midfielder_info()
midfielder_ids = list(midfielder_info.keys()) if midfielder_info else None

print(f"Found {len(midfielder_info)} midfielders")
if midfielder_ids:
    print(f"Sample IDs: {midfielder_ids[:5]}")


Found 91 midfielders
Sample IDs: [2858, 5468, 5521, 6871, 7252]


## Extract Event Features


## Diagnostic: Check Available Event Subtypes


In [4]:
# Load events data to check what subtypes exist
events_df = pd.read_csv(events_file)

# Check on_ball_engagement events
on_ball_engagements = events_df[events_df["event_type"] == "on_ball_engagement"]

print(f"Total on_ball_engagement events: {len(on_ball_engagements)}")
print(f"\nAll unique event_subtype values for on_ball_engagement:")
print(on_ball_engagements["event_subtype"].value_counts())

# Check for midfielders specifically
if midfielder_ids:
    midfielder_on_ball = on_ball_engagements[
        on_ball_engagements["player_id"].isin(midfielder_ids)
    ]
    print(f"\nOn-ball engagements by midfielders: {len(midfielder_on_ball)}")
    print(f"\nEvent subtypes for midfielders:")
    print(midfielder_on_ball["event_subtype"].value_counts())
    
    # Check what the code is looking for vs what exists
    expected_duel_subtypes = ["tackle", "interception", "recovery", "duel"]
    actual_subtypes = midfielder_on_ball["event_subtype"].unique()
    print(f"\nExpected duel subtypes: {expected_duel_subtypes}")
    print(f"Actual subtypes in data: {sorted(actual_subtypes)}")
    print(f"\nMatch? {set(expected_duel_subtypes) & set(actual_subtypes)}")


Total on_ball_engagement events: 937

All unique event_subtype values for on_ball_engagement:
event_subtype
pressing          244
pressure          236
recovery_press    206
other             164
counter_press      87
Name: count, dtype: int64

On-ball engagements by midfielders: 434

Event subtypes for midfielders:
event_subtype
pressing          110
pressure          107
recovery_press    100
other              75
counter_press      42
Name: count, dtype: int64

Expected duel subtypes: ['tackle', 'interception', 'recovery', 'duel']
Actual subtypes in data: ['counter_press', 'other', 'pressing', 'pressure', 'recovery_press']

Match? set()


In [5]:
# Check player_possession end_type values
player_possession = events_df[events_df["event_type"] == "player_possession"]
print("Unique end_type values in player_possession events:")
print(player_possession["end_type"].value_counts())

# Check for midfielders
if midfielder_ids:
    midfielder_possession = player_possession[
        player_possession["player_id"].isin(midfielder_ids)
    ]
    print(f"\nEnd types for midfielders:")
    print(midfielder_possession["end_type"].value_counts())


Unique end_type values in player_possession events:
end_type
pass               902
possession_loss     48
shot                23
foul_suffered       15
clearance            9
unknown              2
Name: count, dtype: int64

End types for midfielders:
end_type
pass               275
possession_loss     20
foul_suffered        8
shot                 6
clearance            2
Name: count, dtype: int64


## Extract Events by Type (OBR, PO, PP, OBE)

The new functionality allows you to extract events separately by type:
- **OBR** = Off-Ball Run
- **PO** = Passing Option  
- **PP** = Player Possession
- **OBE** = On-Ball Engagement


In [6]:
# Extract OBR (Off-Ball Run) events for midfielders
obr_events = extract_obr_events(events_df, player_ids=midfielder_ids, include_attributes=True)

print(f"Total OBR events for midfielders: {len(obr_events)}")
print(f"\nAvailable columns: {list(obr_events.columns)}")
print(f"\nSample OBR events:")
obr_events.head()


Total OBR events for midfielders: 272

Available columns: ['event_id', 'match_id', 'player_id', 'player_name', 'event_type', 'event_type_id', 'event_subtype', 'event_subtype_id', 'channel_id_end', 'channel_end', 'associated_player_possession_event_id', 'associated_player_possession_frame_start', 'player_in_possession_id', 'player_in_possession_name', 'n_simultaneous_runs', 'give_and_go', 'intended_run_behind', 'push_defensive_line', 'break_defensive_line', 'passing_option_at_start', 'n_opponents_ahead_end', 'n_opponents_ahead_start', 'n_opponents_overtaken']

Sample OBR events:


,event_id,match_id,player_id,player_name,event_type,event_type_id,event_subtype,event_subtype_id,channel_id_end,channel_end,...,player_in_possession_name,n_simultaneous_runs,give_and_go,intended_run_behind,push_defensive_line,break_defensive_line,passing_option_at_start,n_opponents_ahead_end,n_opponents_ahead_start,n_opponents_overtaken
10,1_0,1886347,735573,T. Aquilina,off_ball_run,1,pulling_wide,7.0,1,wide_left,...,M. Natta,2.0,False,NaN,NaN,NaN,True,5.0,5.0,0.0
12,1_1,1886347,795507,L. Bayliss,off_ball_run,1,coming_short,2.0,2,half_space_left,...,M. Natta,2.0,False,NaN,NaN,NaN,True,4.0,4.0,0.0
29,1_4,1886347,50951,J. Brimmer,off_ball_run,1,support,9.0,1,wide_left,...,L. Gillion,1.0,False,NaN,NaN,NaN,True,5.0,5.0,0.0
30,1_5,1886347,23418,F. Gallegos,off_ball_run,1,overlap,5.0,1,wide_left,...,L. Gillion,2.0,False,NaN,NaN,NaN,True,0.0,6.0,6.0
107,1_11,1886347,735573,T. Aquilina,off_ball_run,1,pulling_wide,7.0,2,half_space_left,...,A. Šušnjar,2.0,False,NaN,NaN,NaN,False,6.0,5.0,-1.0


In [25]:
# Extract PO (Passing Option) events for midfielders
po_events = extract_po_events(events_df, player_ids=midfielder_ids, include_attributes=True)

print(f"Total PO events for midfielders: {len(po_events)}")
print(f"\nAvailable columns: {list(po_events.columns)}")
print(f"\nSample PO events:")
po_events.head()


Total PO events for midfielders: 966

Available columns: ['event_id', 'match_id', 'player_id', 'player_name', 'event_type', 'event_type_id', 'channel_id_end', 'channel_end', 'associated_player_possession_event_id', 'associated_player_possession_frame_start', 'associated_off_ball_run_event_id', 'associated_off_ball_run_subtype_id', 'associated_off_ball_run_subtype', 'player_in_possession_id', 'player_in_possession_name', 'targeted', 'received', 'received_in_space', 'passing_option_at_player_possession_start', 'peak_passing_option_frame', 'n_simultaneous_passing_options', 'first_line_break_type_id', 'first_line_break_type', 'second_last_line_break_type_id', 'second_last_line_break_type', 'last_line_break_type_id', 'last_line_break_type', 'high_pass', 'n_opponents_ahead_end', 'n_opponents_ahead_start', 'n_opponents_overtaken']

Sample PO events:


,event_id,match_id,player_id,player_name,event_type,event_type_id,channel_id_end,channel_end,associated_player_possession_event_id,associated_player_possession_frame_start,...,first_line_break_type_id,first_line_break_type,second_last_line_break_type_id,second_last_line_break_type,last_line_break_type_id,last_line_break_type,high_pass,n_opponents_ahead_end,n_opponents_ahead_start,n_opponents_overtaken
2,7_0,1886347,735574,K. Grozos,passing_option,7,3,center,8_1,48.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,7_2,1886347,50978,C. Timmins,passing_option,7,2,half_space_left,8_2,72.0,...,2.0,around,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,7_3,1886347,795507,L. Bayliss,passing_option,7,2,half_space_left,8_2,72.0,...,2.0,around,1.0,through,NaN,NaN,NaN,NaN,NaN,NaN
9,7_5,1886347,735573,T. Aquilina,passing_option,7,1,wide_left,8_2,72.0,...,2.0,around,2.0,around,NaN,NaN,NaN,NaN,NaN,NaN
21,7_9,1886347,23418,F. Gallegos,passing_option,7,2,half_space_left,8_4,243.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
# Extract PP (Player Possession) events for midfielders
pp_events = extract_pp_events(events_df, player_ids=midfielder_ids, include_attributes=True)

print(f"Total PP events for midfielders: {len(pp_events)}")
print(f"\nAvailable columns: {list(pp_events.columns)}")
print(f"\nSample PP events:")
pp_events.head()


Total PP events for midfielders: 311

Available columns: ['event_id', 'match_id', 'player_id', 'player_name', 'event_type', 'event_type_id', 'channel_id_end', 'channel_end', 'associated_player_possession_frame_end', 'associated_player_possession_end_type_id', 'associated_player_possession_end_type', 'game_interruption_before_id', 'game_interruption_before', 'game_interruption_after_id', 'game_interruption_after', 'start_type_id', 'start_type', 'end_type_id', 'end_type', 'carry', 'forward_momentum', 'break_defensive_line', 'pass_ahead', 'received_in_space']

Sample PP events:


,event_id,match_id,player_id,player_name,event_type,event_type_id,channel_id_end,channel_end,associated_player_possession_frame_end,associated_player_possession_end_type_id,...,game_interruption_after,start_type_id,start_type,end_type_id,end_type,carry,forward_momentum,break_defensive_line,pass_ahead,received_in_space
39,8_7,1886347,23418,F. Gallegos,player_possession,8,1,wide_left,NaN,NaN,...,NaN,1.0,pass_reception,1.0,pass,False,False,NaN,False,NaN
54,8_12,1886347,23418,F. Gallegos,player_possession,8,5,wide_right,NaN,NaN,...,NaN,1.0,pass_reception,1.0,pass,True,False,NaN,False,NaN
71,8_15,1886347,23418,F. Gallegos,player_possession,8,5,wide_right,NaN,NaN,...,NaN,7.0,throw_in_reception,1.0,pass,True,False,NaN,False,NaN
81,8_16,1886347,50951,J. Brimmer,player_possession,8,5,wide_right,NaN,NaN,...,NaN,1.0,pass_reception,1.0,pass,False,False,NaN,False,NaN
86,8_17,1886347,14736,L. Verstraete,player_possession,8,4,half_space_right,NaN,NaN,...,throw_in_against,1.0,pass_reception,1.0,pass,True,False,NaN,False,NaN


In [9]:
# Extract OBE (On-Ball Engagement) events for midfielders
obe_events = extract_obe_events(events_df, player_ids=midfielder_ids, include_attributes=True)

print(f"Total OBE events for midfielders: {len(obe_events)}")
print(f"\nAvailable columns: {list(obe_events.columns)}")
print(f"\nSample OBE events:")
obe_events.head()


Total OBE events for midfielders: 434

Available columns: ['event_id', 'match_id', 'player_id', 'player_name', 'event_type', 'event_type_id', 'event_subtype', 'event_subtype_id', 'frame_physical_start', 'possession_danger', 'beaten_by_possession', 'beaten_by_movement', 'stop_possession_danger', 'reduce_possession_danger', 'force_backward', 'player_targeted_id', 'player_targeted_name', 'affected_line_breaking_passing_option_id', 'affected_line_break_id', 'affected_line_break', 'affected_line_breaking_passing_option_attempted', 'affected_line_breaking_passing_option_xthreat', 'affected_line_breaking_passing_option_dangerous', 'affected_line_breaking_passing_option_run_subtype_id', 'affected_line_breaking_passing_option_run_subtype', 'pressing_chain', 'pressing_chain_length', 'pressing_chain_end_type_id', 'pressing_chain_end_type', 'pressing_chain_index', 'index_in_pressing_chain']

Sample OBE events:


,event_id,match_id,player_id,player_name,event_type,event_type_id,event_subtype,event_subtype_id,frame_physical_start,possession_danger,...,affected_line_breaking_passing_option_xthreat,affected_line_breaking_passing_option_dangerous,affected_line_breaking_passing_option_run_subtype_id,affected_line_breaking_passing_option_run_subtype,pressing_chain,pressing_chain_length,pressing_chain_end_type_id,pressing_chain_end_type,pressing_chain_index,index_in_pressing_chain
4,9_0,1886347,50951,J. Brimmer,on_ball_engagement,9,pressing,11.0,34.0,False,...,NaN,NaN,NaN,NaN,True,2.0,2.0,disruption,0.0,1.0
18,9_2,1886347,795505,E. Adams,on_ball_engagement,9,pressure,12.0,232.0,False,...,0.0065,False,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN
58,9_7,1886347,50978,C. Timmins,on_ball_engagement,9,counter_press,13.0,549.0,False,...,NaN,NaN,NaN,NaN,True,3.0,2.0,disruption,1.0,1.0
60,9_8,1886347,735573,T. Aquilina,on_ball_engagement,9,counter_press,13.0,566.0,False,...,NaN,NaN,NaN,NaN,True,3.0,2.0,disruption,1.0,2.0
70,9_10,1886347,795507,L. Bayliss,on_ball_engagement,9,recovery_press,14.0,746.0,False,...,NaN,NaN,NaN,NaN,True,3.0,2.0,disruption,2.0,1.0


### Check New Attributes

Let's examine some of the new attributes available in the extracted events:


In [10]:
# Check channel_end and channel_id_end in OBR events
if not obr_events.empty and "channel_end" in obr_events.columns:
    print("Channel distribution for OBR events:")
    print(obr_events["channel_end"].value_counts())
    print(f"\nChannel ID distribution:")
    if "channel_id_end" in obr_events.columns:
        print(obr_events["channel_id_end"].value_counts())

# Check associated_player_possession attributes in OBR events
if not obr_events.empty:
    print("\n" + "="*60)
    print("Associated Player Possession attributes in OBR events:")
    assoc_cols = [col for col in obr_events.columns if "associated_player_possession" in col]
    if assoc_cols:
        print(f"Available columns: {assoc_cols}")
        print(f"\nSample values:")
        print(obr_events[assoc_cols].head())
    else:
        print("No associated_player_possession columns found")


Channel distribution for OBR events:
channel_end
center              70
half_space_right    59
half_space_left     58
wide_left           54
wide_right          31
Name: count, dtype: int64

Channel ID distribution:
channel_id_end
3    70
4    59
2    58
1    54
5    31
Name: count, dtype: int64

Associated Player Possession attributes in OBR events:
Available columns: ['associated_player_possession_event_id', 'associated_player_possession_frame_start']

Sample values:
    associated_player_possession_event_id  \
10                                    8_2   
12                                    8_2   
29                                    8_6   
30                                    8_6   
107                                  8_22   

     associated_player_possession_frame_start  
10                                       72.0  
12                                       72.0  
29                                      301.0  
30                                      301.0  
107            

In [11]:
# Check associated_off_ball_run attributes in PO events
if not po_events.empty:
    print("Associated Off-Ball Run attributes in PO events:")
    assoc_cols = [col for col in po_events.columns if "associated_off_ball_run" in col]
    if assoc_cols:
        print(f"Available columns: {assoc_cols}")
        print(f"\nSample values:")
        print(po_events[assoc_cols].head())
    else:
        print("No associated_off_ball_run columns found")
    
    # Check channel_end
    if "channel_end" in po_events.columns:
        print("\n" + "="*60)
        print("Channel distribution for PO events:")
        print(po_events["channel_end"].value_counts())


Associated Off-Ball Run attributes in PO events:
Available columns: ['associated_off_ball_run_event_id', 'associated_off_ball_run_subtype_id', 'associated_off_ball_run_subtype']

Sample values:
   associated_off_ball_run_event_id  associated_off_ball_run_subtype_id  \
2                               NaN                                 NaN   
6                               NaN                                 NaN   
7                               1_1                                 2.0   
9                               1_0                                 7.0   
21                              NaN                                 NaN   

   associated_off_ball_run_subtype  
2                              NaN  
6                              NaN  
7                     coming_short  
9                     pulling_wide  
21                             NaN  

Channel distribution for PO events:
channel_end
center              248
wide_left           202
half_space_left     196
half_space_r

In [12]:
# Check game_interruption and start_type/end_type in PP events
if not pp_events.empty:
    print("Game Interruption and Start/End Type attributes in PP events:")
    
    # Game interruptions
    game_int_cols = [col for col in pp_events.columns if "game_interruption" in col]
    if game_int_cols:
        print(f"\nGame interruption columns: {game_int_cols}")
        if "game_interruption_before" in pp_events.columns:
            print("\nGame interruption before distribution:")
            print(pp_events["game_interruption_before"].value_counts())
        if "game_interruption_after" in pp_events.columns:
            print("\nGame interruption after distribution:")
            print(pp_events["game_interruption_after"].value_counts())
    
    # Start/End types
    if "start_type" in pp_events.columns:
        print("\n" + "="*60)
        print("Start type distribution:")
        print(pp_events["start_type"].value_counts())
    if "end_type" in pp_events.columns:
        print("\nEnd type distribution:")
        print(pp_events["end_type"].value_counts())
    
    # Channel end
    if "channel_end" in pp_events.columns:
        print("\n" + "="*60)
        print("Channel distribution for PP events:")
        print(pp_events["channel_end"].value_counts())


Game Interruption and Start/End Type attributes in PP events:

Game interruption columns: ['game_interruption_before_id', 'game_interruption_before', 'game_interruption_after_id', 'game_interruption_after']

Game interruption before distribution:
game_interruption_before
throw_in_for         24
free_kick_for         5
corner_for            4
throw_in_against      3
corner_against        1
goal_kick_for         1
free_kick_against     1
Name: count, dtype: int64

Game interruption after distribution:
game_interruption_after
throw_in_against     11
free_kick_for         8
throw_in_for          7
goal_kick_against     5
free_kick_against     5
corner_for            1
Name: count, dtype: int64

Start type distribution:
start_type
pass_reception         207
recovery                26
throw_in_reception      23
pass_interception       22
keep_possession         16
unknown                  8
free_kick_reception      5
corner_reception         3
goal_kick_reception      1
Name: count, dtype: i

## Extract Aggregated Event Features


In [13]:
# Check off_ball_run events and their subtypes
off_ball_runs = events_df[events_df["event_type"] == "off_ball_run"]

print(f"Total off_ball_run events: {len(off_ball_runs)}")
print(f"\nAll unique event_subtype values for off_ball_run:")
print(off_ball_runs["event_subtype"].value_counts())

# Check for midfielders specifically
if midfielder_ids:
    midfielder_off_ball_runs = off_ball_runs[
        off_ball_runs["player_id"].isin(midfielder_ids)
    ]
    print(f"\nOff-ball runs by midfielders: {len(midfielder_off_ball_runs)}")
    print(f"\nEvent subtypes for midfielders:")
    print(midfielder_off_ball_runs["event_subtype"].value_counts())
    
    # Check for all expected subtypes
    expected_subtypes = [
        "behind", "coming_short", "cross_receiver", "dropping_off", 
        "overlap", "pulling_half_space", "pulling_wide", 
        "run_ahead_of_the_ball", "support", "underlap"
    ]
    actual_subtypes = midfielder_off_ball_runs["event_subtype"].unique()
    print(f"\nExpected subtypes: {expected_subtypes}")
    print(f"Actual subtypes in data: {sorted(actual_subtypes)}")
    print(f"\nMatch? {set(expected_subtypes) & set(actual_subtypes)}")
    print(f"Missing? {set(expected_subtypes) - set(actual_subtypes)}")


Total off_ball_run events: 599

All unique event_subtype values for off_ball_run:
event_subtype
run_ahead_of_the_ball    147
coming_short             100
dropping_off              77
support                   74
cross_receiver            61
pulling_wide              42
behind                    42
pulling_half_space        21
underlap                  20
overlap                   15
Name: count, dtype: int64

Off-ball runs by midfielders: 272

Event subtypes for midfielders:
event_subtype
coming_short             76
run_ahead_of_the_ball    63
support                  40
dropping_off             21
behind                   16
pulling_wide             14
cross_receiver           14
pulling_half_space       13
underlap                 11
overlap                   4
Name: count, dtype: int64

Expected subtypes: ['behind', 'coming_short', 'cross_receiver', 'dropping_off', 'overlap', 'pulling_half_space', 'pulling_wide', 'run_ahead_of_the_ball', 'support', 'underlap']
Actual subtypes in dat

In [14]:
# Extract event features
features_df = extract_event_features_from_file(
    events_file=events_file,
    midfielder_ids=midfielder_ids,
)

print(f"Extracted features for {len(features_df)} player-match combinations")
print(f"\nColumns: {list(features_df.columns)}")
features_df.head()


Extracted features for 8 player-match combinations

Columns: ['player_id', 'match_id', 'carries_count', 'carries_distance', 'passes_into_space', 'pressures', 'progressive_actions', 'pressing_count', 'pressure_count', 'recovery_press_count', 'counter_press_count', 'off_ball_runs_count', 'off_ball_runs_behind', 'off_ball_runs_coming_short', 'off_ball_runs_cross_receiver', 'off_ball_runs_overlap', 'off_ball_runs_run_ahead', 'off_ball_runs_support', 'off_ball_runs_underlap', 'passing_options_count', 'passing_options_targeted', 'passing_options_received']


/Users/arvid/development/personal/analytics_cup/analytics_cup_analyst/src/movement_dna/event_features.py:663: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  pressures = on_ball_engagements[
/Users/arvid/development/personal/analytics_cup/analytics_cup_analyst/src/movement_dna/event_features.py:663: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  pressures = on_ball_engagements[
/Users/arvid/development/personal/analytics_cup/analytics_cup_analyst/src/movement_dna/event_features.py:663: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  pressures = on_ball_engagements[
/Users/arvid/development/personal/analytics_cup/analytics_cup_analyst/src/movement_dna/event_features.py:663: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  pressures = on_ball_engagements[
/Users/arvid/development/personal/analytics_cup/analytics_cup_analyst/src/movement_dna/event_features.py:663: UserWa

,player_id,match_id,carries_count,carries_distance,passes_into_space,pressures,progressive_actions,pressing_count,pressure_count,recovery_press_count,...,off_ball_runs_behind,off_ball_runs_coming_short,off_ball_runs_cross_receiver,off_ball_runs_overlap,off_ball_runs_run_ahead,off_ball_runs_support,off_ball_runs_underlap,passing_options_count,passing_options_targeted,passing_options_received
0,23418,1886347,23,224.66,56,54,0,20,10,16,...,1,14,0,2,11,7,6,152,43,40
1,50951,1886347,17,149.17,31,40,0,15,7,10,...,4,4,7,1,9,5,0,138,37,28
2,14736,1886347,32,276.96,65,34,1,10,9,12,...,0,9,1,0,3,7,3,128,27,25
3,50978,1886347,17,136.54,38,33,2,6,11,11,...,0,22,0,0,4,3,0,118,30,26
4,735574,1886347,20,114.74,49,28,0,6,12,9,...,0,13,1,1,3,1,0,118,32,32


## Add Player Names


In [15]:
# Merge with player names
if midfielder_info:
    features_df["player_name"] = features_df["player_id"].map(
        lambda x: midfielder_info.get(x, {}).get("player_name", f"Player {x}")
    )
    features_df["team_name"] = features_df["player_id"].map(
        lambda x: midfielder_info.get(x, {}).get("team_name", "Unknown")
    )

features_df[["player_name", "team_name", "carries_count", "pressures", "off_ball_runs_count"]].head(10)


,player_name,team_name,carries_count,pressures,off_ball_runs_count
0,Luis Felipe Gallegos Leiva,Auckland FC,23,54,48
1,Jake Brimmer,Auckland FC,17,40,38
2,Louis Verstraete,Auckland FC,32,34,29
3,Callum Timmins,Newcastle United Jets FC,17,33,30
4,Kosta Grozos,Newcastle United Jets FC,20,28,20
5,Thomas Aquilina,Newcastle United Jets FC,10,56,49
6,Eli Adams,Newcastle United Jets FC,18,61,36
7,Lachlan Bayliss,Newcastle United Jets FC,6,53,22


## Summary Statistics


In [16]:
print("📊 Basic Event Statistics:")
print(f"   Total carries: {features_df['carries_count'].sum()}")
print(f"   Total carry distance: {features_df['carries_distance'].sum():.1f} meters")
print(f"   Total passes into space: {features_df['passes_into_space'].sum()}")
print(f"   Total pressures: {features_df['pressures'].sum()}")
print(f"   Total progressive actions: {features_df['progressive_actions'].sum()}")
print(f"   Total off-ball runs: {features_df['off_ball_runs_count'].sum()}")
print(f"   Total passing options: {features_df['passing_options_count'].sum()}")


📊 Basic Event Statistics:
   Total carries: 143
   Total carry distance: 1202.8 meters
   Total passes into space: 287
   Total pressures: 359
   Total progressive actions: 8
   Total off-ball runs: 272
   Total passing options: 966


## Top Players by Event Type

### Top 5 by Carries


In [17]:
top_carries = features_df.nlargest(5, "carries_count")[
    ["player_name", "team_name", "carries_count", "carries_distance"]
]
top_carries


,player_name,team_name,carries_count,carries_distance
2,Louis Verstraete,Auckland FC,32,276.96
0,Luis Felipe Gallegos Leiva,Auckland FC,23,224.66
4,Kosta Grozos,Newcastle United Jets FC,20,114.74
6,Eli Adams,Newcastle United Jets FC,18,187.98
1,Jake Brimmer,Auckland FC,17,149.17


### Top 5 by Pressures (with subtypes)


In [18]:
top_pressures = features_df.nlargest(5, "pressures")[
    [
        "player_name",
        "team_name",
        "pressures",
        "pressing_count",
        "pressure_count",
        "recovery_press_count",
        "counter_press_count",
    ]
]
top_pressures


,player_name,team_name,pressures,pressing_count,pressure_count,recovery_press_count,counter_press_count
6,Eli Adams,Newcastle United Jets FC,61,15,30,13,3
5,Thomas Aquilina,Newcastle United Jets FC,56,16,14,20,6
0,Luis Felipe Gallegos Leiva,Auckland FC,54,20,10,16,8
7,Lachlan Bayliss,Newcastle United Jets FC,53,22,14,9,8
1,Jake Brimmer,Auckland FC,40,15,7,10,8


### Top 5 by Off-Ball Runs (with subtypes)


In [19]:
top_runs = features_df.nlargest(5, "off_ball_runs_count")[
    [
        "player_name",
        "team_name",
        "off_ball_runs_count",
        "off_ball_runs_behind",
        "off_ball_runs_coming_short",
        "off_ball_runs_cross_receiver",
        "off_ball_runs_overlap",
        "off_ball_runs_run_ahead",
        "off_ball_runs_support",
        "off_ball_runs_underlap",
    ]
]
top_runs


,player_name,team_name,off_ball_runs_count,off_ball_runs_behind,off_ball_runs_coming_short,off_ball_runs_cross_receiver,off_ball_runs_overlap,off_ball_runs_run_ahead,off_ball_runs_support,off_ball_runs_underlap
5,Thomas Aquilina,Newcastle United Jets FC,49,5,4,2,0,14,7,2
0,Luis Felipe Gallegos Leiva,Auckland FC,48,1,14,0,2,11,7,6
1,Jake Brimmer,Auckland FC,38,4,4,7,1,9,5,0
6,Eli Adams,Newcastle United Jets FC,36,5,5,1,0,12,6,0
3,Callum Timmins,Newcastle United Jets FC,30,0,22,0,0,4,3,0


### Top 5 by Progressive Actions


In [20]:
top_progressive = features_df.nlargest(40, "progressive_actions")[
    ["player_name", "team_name", "progressive_actions", "passes_into_space"]
]
top_progressive


,player_name,team_name,progressive_actions,passes_into_space
6,Eli Adams,Newcastle United Jets FC,4,17
3,Callum Timmins,Newcastle United Jets FC,2,38
2,Louis Verstraete,Auckland FC,1,65
7,Lachlan Bayliss,Newcastle United Jets FC,1,12
0,Luis Felipe Gallegos Leiva,Auckland FC,0,56
1,Jake Brimmer,Auckland FC,0,31
4,Kosta Grozos,Newcastle United Jets FC,0,49
5,Thomas Aquilina,Newcastle United Jets FC,0,19


### Top 5 by Passing Options


In [21]:
top_options = features_df.nlargest(5, "passing_options_count")[
    [
        "player_name",
        "team_name",
        "passing_options_count",
        "passing_options_targeted",
        "passing_options_received",
    ]
]
top_options


,player_name,team_name,passing_options_count,passing_options_targeted,passing_options_received
0,Luis Felipe Gallegos Leiva,Auckland FC,152,43,40
1,Jake Brimmer,Auckland FC,138,37,28
2,Louis Verstraete,Auckland FC,128,27,25
5,Thomas Aquilina,Newcastle United Jets FC,127,31,25
3,Callum Timmins,Newcastle United Jets FC,118,30,26


## Detailed View: Individual Player


In [22]:
# Select a player to view in detail
player_idx = 0  # Change this to view different players

player = features_df.iloc[player_idx]

print(f"Player: {player.get('player_name', 'Unknown')}")
print(f"Team: {player.get('team_name', 'Unknown')}")
print(f"Match ID: {int(player['match_id'])}")
print()

print("Event Breakdown:")
print(f"  Carries: {int(player['carries_count'])} ({player['carries_distance']:.1f}m)")
print(f"  Passes into space: {int(player['passes_into_space'])}")
print(f"  Progressive actions: {int(player['progressive_actions'])}")
print()

print("Pressures:")
print(f"  Total: {int(player['pressures'])}")
print(f"    - Pressing: {int(player['pressing_count'])}")
print(f"    - Pressure: {int(player['pressure_count'])}")
print(f"    - Recovery press: {int(player['recovery_press_count'])}")
print(f"    - Counter press: {int(player['counter_press_count'])}")
print()

print("Off-Ball Runs:")
print(f"  Total: {int(player['off_ball_runs_count'])}")
print(f"    - Behind: {int(player['off_ball_runs_behind'])}")
print(f"    - Coming short: {int(player['off_ball_runs_coming_short'])}")
print(f"    - Cross receiver: {int(player['off_ball_runs_cross_receiver'])}")
print(f"    - Overlap: {int(player['off_ball_runs_overlap'])}")
print(f"    - Run ahead: {int(player['off_ball_runs_run_ahead'])}")
print(f"    - Support: {int(player['off_ball_runs_support'])}")
print(f"    - Underlap: {int(player['off_ball_runs_underlap'])}")
print()

print("Passing Options:")
print(f"  Total: {int(player['passing_options_count'])}")
print(f"    - Targeted: {int(player['passing_options_targeted'])}")
print(f"    - Received: {int(player['passing_options_received'])}")


Player: Luis Felipe Gallegos Leiva
Team: Auckland FC
Match ID: 1886347

Event Breakdown:
  Carries: 23 (224.7m)
  Passes into space: 56
  Progressive actions: 0

Pressures:
  Total: 54
    - Pressing: 20
    - Pressure: 10
    - Recovery press: 16
    - Counter press: 8

Off-Ball Runs:
  Total: 48
    - Behind: 1
    - Coming short: 14
    - Cross receiver: 0
    - Overlap: 2
    - Run ahead: 11
    - Support: 7
    - Underlap: 6

Passing Options:
  Total: 152
    - Targeted: 43
    - Received: 40


## All Players Summary Table


In [23]:
# Display summary table of all players
summary_cols = [
    "player_name",
    "team_name",
    "carries_count",
    "pressures",
    "off_ball_runs_count",
    "progressive_actions",
    "passing_options_count",
]

features_df[summary_cols].sort_values("carries_count", ascending=False)


,player_name,team_name,carries_count,pressures,off_ball_runs_count,progressive_actions,passing_options_count
2,Louis Verstraete,Auckland FC,32,34,29,1,128
0,Luis Felipe Gallegos Leiva,Auckland FC,23,54,48,0,152
4,Kosta Grozos,Newcastle United Jets FC,20,28,20,0,118
6,Eli Adams,Newcastle United Jets FC,18,61,36,4,107
1,Jake Brimmer,Auckland FC,17,40,38,0,138
3,Callum Timmins,Newcastle United Jets FC,17,33,30,2,118
5,Thomas Aquilina,Newcastle United Jets FC,10,56,49,0,127
7,Lachlan Bayliss,Newcastle United Jets FC,6,53,22,1,78


## Save Results


In [24]:
# Save results to CSV
output_file = Path.cwd() / "event_features_results.csv"
# Remove player_name and team_name before saving (they're just for display)
save_df = features_df.drop(columns=["player_name", "team_name"], errors="ignore")
save_df.to_csv(output_file, index=False)
print(f"💾 Results saved to: {output_file}")


💾 Results saved to: /Users/arvid/development/personal/analytics_cup/analytics_cup_analyst/event_features_results.csv


,player_id,match_id,carries_count,carries_distance,passes_into_space,pressures,progressive_actions,pressing_count,pressure_count,recovery_press_count,...,off_ball_runs_cross_receiver,off_ball_runs_overlap,off_ball_runs_run_ahead,off_ball_runs_support,off_ball_runs_underlap,passing_options_count,passing_options_targeted,passing_options_received,player_name,team_name
0,23418,1886347,23,224.66,56,54,0,20,10,16,...,0,2,11,7,6,152,43,40,Luis Felipe Gallegos Leiva,Auckland FC
1,50951,1886347,17,149.17,31,40,0,15,7,10,...,7,1,9,5,0,138,37,28,Jake Brimmer,Auckland FC
2,14736,1886347,32,276.96,65,34,1,10,9,12,...,1,0,3,7,3,128,27,25,Louis Verstraete,Auckland FC
3,50978,1886347,17,136.54,38,33,2,6,11,11,...,0,0,4,3,0,118,30,26,Callum Timmins,Newcastle United Jets FC
4,735574,1886347,20,114.74,49,28,0,6,12,9,...,1,1,3,1,0,118,32,32,Kosta Grozos,Newcastle United Jets FC
5,735573,1886347,10,72.84,19,56,0,16,14,20,...,2,0,14,7,2,127,31,25,Thomas Aquilina,Newcastle United Jets FC
6,795505,1886347,18,187.98,17,61,4,15,30,13,...,1,0,12,6,0,107,38,28,Eli Adams,Newcastle United Jets FC
7,795507,1886347,6,39.88,12,53,1,22,14,9,...,2,0,7,4,0,78,18,10,Lachlan Bayliss,Newcastle United Jets FC
